In [9]:
# 본 실습자료는 ASR 커널을 활용해 주세요.
# 본 실습자료는 ssh 포트포워딩 후 http://127.0.0.1:port2/jupyter/hub/login 로 접속해 주세요.
## port2: jupyter notebook 용 포트

In [10]:
!ip route | head

default via 172.17.0.1 dev eth0 
172.17.0.0/16 dev eth0 proto kernel scope link src 172.17.0.2 


In [11]:
pip install -q -U ipywidgets pandas

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os, base64, subprocess
from pathlib import Path
import pandas as pd
import ipywidgets as W
from IPython.display import display, HTML
import uuid

# ─────────────────────────────────────────────
# 0) 세션별 녹음 위젯 식별자 (셀 재실행 시 ID 충돌 방지)
# ─────────────────────────────────────────────
REC_ID = f"rec_{uuid.uuid4().hex}"

# ─────────────────────────────────────────────
# 1) 저장 경로(서버/환경에 맞게 여기만 바꾸면 전체 반영됨)
# ─────────────────────────────────────────────
HOME = Path.home()
ROOT = HOME / "data_team_tts"
AUDIO_DIR = ROOT / "audio"       # 오디오 저장 루트: ~/data_team/audio/student01/0001.wav ...
REFS_CSV = ROOT / "refs.csv"     # 정답 텍스트(스크립트) 저장 CSV

# ─────────────────────────────────────────────
# 2) refs.csv 로드/초기화
#    - file: "student01/0001.wav" 같은 상대경로
#    - transcript: 해당 음성의 정답 텍스트(스크립트)
# ─────────────────────────────────────────────
def load_refs():
    if REFS_CSV.exists():
        df = pd.read_csv(REFS_CSV)
    else:
        df = pd.DataFrame(columns=["file", "transcript"])
        REFS_CSV.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(REFS_CSV, index=False)

    if "file" not in df.columns:
        df["file"] = ""
    if "transcript" not in df.columns:
        df["transcript"] = ""

    return df

def save_refs(df: pd.DataFrame):
    REFS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(REFS_CSV, index=False)

df = load_refs()

In [13]:
# ─────────────────────────────────────────────
# 3) 학생 폴더 목록 가져오기
#    - ~/data_team/audio 하위 폴더명을 student 목록으로 사용
# ─────────────────────────────────────────────
def list_students():
    if not AUDIO_DIR.exists():
        return []
    return sorted([
        p.name
        for p in AUDIO_DIR.iterdir()
        if p.is_dir() and not p.name.startswith(".")
    ])

students = list_students() or ["student01"]

# ─────────────────────────────────────────────
# 4) UI 위젯 구성(왼쪽: 학생/목록, 오른쪽: 텍스트/녹음/저장)
# ─────────────────────────────────────────────
student_dd = W.Dropdown(
    options=students,
    value=students[0],
    description="Student:",
    layout=W.Layout(width="320px")
)

rows_sel = W.Select(
    options=[],
    description="Rows:",
    layout=W.Layout(width="420px", height="260px")
)

transcript_ta = W.Textarea(
    description="Text:",
    layout=W.Layout(width="720px", height="140px")
)

status_html = W.HTML()

btn_save_text = W.Button(description="Save Text", button_style="primary")
btn_add = W.Button(description="Add Row")
btn_delete = W.Button(description="Delete Row", button_style="danger")
btn_refresh = W.Button(description="Refresh")

In [14]:
# ─────────────────────────────────────────────
# 5) 녹음 데이터(base64)를 파이썬으로 전달하기 위한 숨은 textarea
#    - JS가 여기 textarea에 base64 문자열을 넣어줌
# ─────────────────────────────────────────────
audio_b64 = W.Textarea(
    value="",
    layout=W.Layout(width="1px", height="1px", opacity="0", margin="0", padding="0")
)

btn_save_audio = W.Button(description="Save Recording", button_style="success")
btn_convert_wav = W.Button(description="Convert to WAV", button_style="warning")

recorder_out = W.Output()

# ─────────────────────────────────────────────
# 6) HTML/JS Recorder UI
#    - Start → Stop 하면 16k mono WAV로 리샘플링 후 base64로 hidden textarea에 채움
# ─────────────────────────────────────────────
with recorder_out:
    display(HTML(f"""
<div id="{REC_ID}" style="border:1px solid #ddd; padding:12px; border-radius:8px; background:#fafafa;">
  <div style="font-weight:600; margin-bottom:8px;">Recording (WAV 16k mono)</div>

  <div style="display:flex; gap:8px; align-items:center; margin:8px 0;">
    <button id="{REC_ID}_btnStart">● Start</button>
    <button id="{REC_ID}_btnStop" disabled>■ Stop</button>
    <span id="{REC_ID}_state" style="margin-left:6px; color:#555;"></span>
  </div>

  <audio id="{REC_ID}_player" controls style="width:100%; margin-top:6px;"></audio>
  <pre id="{REC_ID}_log" style="background:#111; color:#eee; padding:10px; border-radius:8px; white-space:pre-wrap; margin-top:8px;"></pre>
</div>

<script>
(() => {{
  const root = document.getElementById("{REC_ID}");
  if (!root) return;
  if (root.dataset.inited === "1") return;
  root.dataset.inited = "1";

  const btnStart = document.getElementById("{REC_ID}_btnStart");
  const btnStop  = document.getElementById("{REC_ID}_btnStop");
  const stateEl  = document.getElementById("{REC_ID}_state");
  const player   = document.getElementById("{REC_ID}_player");
  const logEl    = document.getElementById("{REC_ID}_log");

  const log = (msg) => {{
    logEl.textContent += msg + "\\n";
    logEl.scrollTop = logEl.scrollHeight;
  }};

  let stream = null;
  let mediaRecorder = null;
  let chunks = [];
  let wavBlob = null;

  function floatTo16BitPCM(view, input) {{
    for (let i = 0; i < input.length; i++) {{
      let s = Math.max(-1, Math.min(1, input[i]));
      view.setInt16(44 + i * 2, s < 0 ? s * 0x8000 : s * 0x7FFF, true);
    }}
  }}

  function writeString(view, offset, string) {{
    for (let i = 0; i < string.length; i++) {{
      view.setUint8(offset + i, string.charCodeAt(i));
    }}
  }}

  function encodeWAV(samples, sampleRate) {{
    const buffer = new ArrayBuffer(44 + samples.length * 2);
    const view = new DataView(buffer);

    writeString(view, 0, "RIFF");
    view.setUint32(4, 36 + samples.length * 2, true);
    writeString(view, 8, "WAVE");
    writeString(view, 12, "fmt ");
    view.setUint32(16, 16, true);
    view.setUint16(20, 1, true);
    view.setUint16(22, 1, true);           // mono
    view.setUint32(24, sampleRate, true);
    view.setUint32(28, sampleRate * 2, true);
    view.setUint16(32, 2, true);
    view.setUint16(34, 16, true);
    writeString(view, 36, "data");
    view.setUint32(40, samples.length * 2, true);

    floatTo16BitPCM(view, samples);
    return new Blob([view], {{ type: "audio/wav" }});
  }}

  async function resampleTo16k(anyAudioBlob) {{
    const ab = await anyAudioBlob.arrayBuffer();
    const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
    const audioBuffer = await audioCtx.decodeAudioData(ab);

    const targetRate = 16000;
    const offline = new OfflineAudioContext(1, Math.ceil(audioBuffer.duration * targetRate), targetRate);

    const source = offline.createBufferSource();
    source.buffer = audioBuffer;
    source.connect(offline.destination);
    source.start(0);

    const rendered = await offline.startRendering();
    const ch0 = rendered.getChannelData(0);
    return encodeWAV(ch0, targetRate);
  }}

  async function blobToB64(blob) {{
    const buf = await blob.arrayBuffer();
    let binary = "";
    const bytes = new Uint8Array(buf);
    const chunkSize = 0x8000;
    for (let i = 0; i < bytes.length; i += chunkSize) {{
      binary += String.fromCharCode(...bytes.subarray(i, i + chunkSize));
    }}
    return btoa(binary);
  }}

  function setState(msg) {{
    stateEl.textContent = msg;
  }}

  btnStart.onclick = async () => {{
    try {{
      if (!navigator.mediaDevices?.getUserMedia) {{
        setState("getUserMedia not available");
        return;
      }}

      wavBlob = null;
      chunks = [];

      stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
      mediaRecorder = new MediaRecorder(stream);

      mediaRecorder.ondataavailable = (e) => {{
        if (e.data && e.data.size > 0) chunks.push(e.data);
      }};

      mediaRecorder.onstop = async () => {{
        try {{
          const raw = new Blob(chunks, {{ type: chunks[0]?.type || "audio/webm" }});
          log("raw: " + raw.type + ", " + raw.size + " bytes");

          wavBlob = await resampleTo16k(raw);
          log("wav16k: " + wavBlob.size + " bytes");

          player.src = URL.createObjectURL(wavBlob);

          const b64 = await blobToB64(wavBlob);

          // hidden textarea(ipywidgets) 찾기: rec-b64-{REC_ID}
          const ta = document.querySelector('.rec-b64-{REC_ID} textarea');
          if (ta) {{
            ta.value = b64;
            ta.dispatchEvent(new Event("input", {{ bubbles: true }}));
            ta.dispatchEvent(new Event("change", {{ bubbles: true }}));
            setState("Recorded (base64 filled)");
          }} else {{
            setState("Recorded but hidden textarea not found");
          }}
        }} catch (err) {{
          console.error(err);
          setState("Stop done, but convert failed");
        }}
      }};

      mediaRecorder.start();
      setState("Recording…");

      btnStart.disabled = true;
      btnStop.disabled = false;
      btnStop.removeAttribute("disabled");

    }} catch (err) {{
      console.error(err);
      setState("Mic error: " + (err?.name || err));
    }}
  }};

  btnStop.onclick = () => {{
    try {{
      if (mediaRecorder && mediaRecorder.state !== "inactive") {{
        mediaRecorder.stop();
      }}
      if (stream) stream.getTracks().forEach(t => t.stop());

      btnStop.disabled = true;
      btnStart.disabled = false;
      setState("Stopping…");
    }} catch (err) {{
      console.error(err);
      setState("Stop error");
    }}
  }};
}})();
</script>
"""))

# ─────────────────────────────────────────────
# 7) JS가 찾아서 base64를 꽂을 수 있도록 hidden textarea에 클래스 부여
# ─────────────────────────────────────────────
audio_b64.add_class(f"rec-b64-{REC_ID}")
audio_b64._dom_classes = tuple(list(audio_b64._dom_classes) + [f"rec-b64-{REC_ID}"])

In [15]:
# ─────────────────────────────────────────────
# 8) 목록 표시용 라벨(파일 존재 여부 표시)
#    - wav 있으면 ✅ / webm만 있으면 🟡 / 둘 다 없으면 ❌
# ─────────────────────────────────────────────
def row_label(relpath):
    wav = AUDIO_DIR / relpath
    webm = wav.with_suffix(".webm")
    exists = "✅" if wav.exists() else ("🟡" if webm.exists() else "❌")
    return f"{exists} {relpath}"

# ─────────────────────────────────────────────
# 9) 현재 student 기준으로 rows_sel 목록 새로고침
# ─────────────────────────────────────────────
def refresh_rows():
    global df
    df = load_refs()

    st = student_dd.value
    sub = df[df["file"].astype(str).str.startswith(f"{st}/")].copy()

    opts = []
    for idx, r in sub.iterrows():
        opts.append((row_label(r["file"]), idx))

    rows_sel.options = opts
    transcript_ta.value = ""
    status_html.value = f"<div style='color:#666;'>Loaded {len(sub)} rows for <b>{st}</b></div>"

# ─────────────────────────────────────────────
# 10) 다음 파일명 자동 생성: student01/0001.wav → 0002.wav ...
# ─────────────────────────────────────────────
def ensure_next_filename(student):
    sub = df[df["file"].astype(str).str.startswith(f"{student}/")]["file"].astype(str)
    nums = []
    for p in sub:
        try:
            stem = Path(p).stem
            nums.append(int(stem))
        except:
            pass
    nxt = (max(nums) + 1) if nums else 1
    return f"{student}/{nxt:04d}.wav"

# ─────────────────────────────────────────────
# 11) rows_sel 선택 시, transcript textarea에 해당 텍스트 로드
# ─────────────────────────────────────────────
def on_select_change(change):
    if change["new"] is None:
        return
    idx = change["new"]
    transcript_ta.value = str(df.loc[idx, "transcript"]) if idx in df.index else ""
    status_html.value = f"<div style='color:#444;'>Selected: <code>{df.loc[idx,'file']}</code></div>"

rows_sel.observe(on_select_change, names="value")

# ─────────────────────────────────────────────
# 12) 텍스트 저장(Save Text)
# ─────────────────────────────────────────────
def on_save_text(_):
    idx = rows_sel.value
    if idx is None:
        status_html.value = "<div style='color:#b00;'>Select a row first.</div>"
        return
    df.loc[idx, "transcript"] = transcript_ta.value.strip()
    save_refs(df)
    status_html.value = "<div style='color:#0a0;'>Text saved to refs.csv</div>"

# ─────────────────────────────────────────────
# 13) 행 추가(Add Row) / 삭제(Delete Row) / 갱신(Refresh)
# ─────────────────────────────────────────────
def on_add(_):
    st = student_dd.value
    rel = ensure_next_filename(st)
    new_row = pd.DataFrame([{"file": rel, "transcript": ""}])
    new_df = pd.concat([df, new_row], ignore_index=True)
    save_refs(new_df)
    refresh_rows()
    status_html.value = f"<div style='color:#0a0;'>Added: <code>{rel}</code></div>"

def on_delete(_):
    idx = rows_sel.value
    if idx is None:
        status_html.value = "<div style='color:#b00;'>Select a row first.</div>"
        return
    rel = str(df.loc[idx, "file"])
    df2 = df.drop(index=idx)
    save_refs(df2)
    refresh_rows()
    status_html.value = f"<div style='color:#0a0;'>Deleted row: <code>{rel}</code></div>"

def on_refresh(_):
    refresh_rows()

btn_save_text.on_click(on_save_text)
btn_add.on_click(on_add)
btn_delete.on_click(on_delete)
btn_refresh.on_click(on_refresh)

# student 변경 시 목록 갱신
def on_student_change(change):
    refresh_rows()

student_dd.observe(on_student_change, names="value")

# 초기 로드
refresh_rows()

In [16]:
# ─────────────────────────────────────────────
# 14) 녹음(base64) → wav 파일로 저장
#    - Start/Stop 후 audio_b64에 base64가 들어있어야 저장 가능
# ─────────────────────────────────────────────
def on_save_audio(_):
    idx = rows_sel.value
    if idx is None:
        status_html.value = "<div style='color:#b00;'>Select a row first.</div>"
        return

    b64 = audio_b64.value.strip()
    if not b64:
        status_html.value = "<div style='color:#b00;'>No recording yet. Click Start → Stop first.</div>"
        return

    rel = str(df.loc[idx, "file"])
    out_wav = AUDIO_DIR / rel
    out_wav.parent.mkdir(parents=True, exist_ok=True)

    out_wav.write_bytes(base64.b64decode(b64))
    status_html.value = f"<div style='color:#0a0;'>Saved WAV: <code>{out_wav}</code></div>"
    refresh_rows()

# ─────────────────────────────────────────────
# 15) (옵션) ffmpeg 변환: webm -> wav
#    - 이 함수는 "서버에 webm 파일이 이미 저장돼 있을 때" 변환하는 용도
#    - 현재 로직은 base64에서 바로 wav 저장하므로, 수업에서는 '참고'로 둠
# ─────────────────────────────────────────────
def on_convert_wav(_):
    idx = rows_sel.value
    if idx is None:
        status_html.value = "<div style='color:#b00;'>Select a row first.</div>"
        return

    rel = str(df.loc[idx, "file"])
    wav_path = AUDIO_DIR / rel
    webm_path = wav_path.with_suffix(".webm")

    if not webm_path.exists():
        status_html.value = "<div style='color:#b00;'>No .webm file. Save recording first.</div>"
        return

    wav_path.parent.mkdir(parents=True, exist_ok=True)

    cmd = ["ffmpeg", "-y", "-i", str(webm_path), "-ac", "1", "-ar", "16000", str(wav_path)]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        status_html.value = f"<div style='color:#0a0;'>Converted to WAV: <code>{wav_path}</code></div>"
        refresh_rows()
    except Exception as e:
        status_html.value = f"<div style='color:#b00;'>ffmpeg failed: {e}</div>"

btn_save_audio.on_click(on_save_audio)
btn_convert_wav.on_click(on_convert_wav)

# ─────────────────────────────────────────────
# 16) 최종 레이아웃 표시
# ─────────────────────────────────────────────
left = W.VBox([
    student_dd,
    rows_sel,
    W.HBox([btn_add, btn_delete, btn_refresh])
])

right = W.VBox([
    transcript_ta,
    W.HBox([btn_save_text]),
    recorder_out,
    audio_b64,  # hidden textarea (JS가 base64를 여기에 넣음)
    W.HBox([btn_save_audio]),
    status_html
])

display(W.HBox([left, right], layout=W.Layout(gap="18px")))